In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_df = spark.table(
    "high_garden.silver.coffee_consumption"
)

display(silver_df.limit(10))

In [0]:
print("Rows:", silver_df.count())

print(
    "Countries:",
    silver_df.select("country").distinct().count()
)

print(
    "Periods:",
    silver_df.select("crop_year").distinct().count()
)

In [0]:
global_trend = (
    silver_df
    .groupBy("start_year")
    .agg(
        F.sum("domestic_consumption")
        .alias("global_consumption")
    )
    .orderBy("start_year")
)

display(global_trend)

Databricks visualization. Run in Databricks to view.

In [0]:
global_window = (
    Window
    .orderBy("start_year")
)

In [0]:
global_growth = (
    global_trend
    .withColumn(
        "previous_consumption",
        F.lag(
            "global_consumption",
            1
        ).over(global_window)
    )
    .withColumn(
        "yoy_growth_pct",
        (
            (
                F.col("global_consumption")
                - F.col("previous_consumption")
            )
            /
            F.col("previous_consumption")
        ) * 100
    )
)

display(global_growth)

Databricks visualization. Run in Databricks to view.

In [0]:
latest_year = (
    silver_df
    .agg(
        F.max("start_year")
        .alias("latest_year")
    )
    .first()["latest_year"]
)

print("Latest year:", latest_year)

In [0]:
latest_market = (
    silver_df
    .filter(
        F.col("start_year") == latest_year
    )
    .select(
        "country",
        "coffee_type",
        "domestic_consumption"
    )
    .orderBy(
        F.desc("domestic_consumption")
    )
)

top_15_market = latest_market.limit(15)

display(top_15_market)

Databricks visualization. Run in Databricks to view.

In [0]:
top_5_countries = [
    row["country"]
    for row in latest_market
    .limit(5)
    .collect()
]

print(top_5_countries)

In [0]:
top_market_trends = (
    silver_df
    .filter(
        F.col("country").isin(top_5_countries)
    )
    .select(
        "country",
        "start_year",
        "domestic_consumption"
    )
    .orderBy(
        "start_year"
    )
)

display(top_market_trends)

Databricks visualization. Run in Databricks to view.

In [0]:
coffee_type_market = (
    silver_df
    .filter(
        F.col("start_year") == latest_year
    )
    .groupBy(
        "coffee_type"
    )
    .agg(
        F.sum("domestic_consumption")
        .alias("total_consumption")
    )
    .orderBy(
        F.desc("total_consumption")
    )
)

display(coffee_type_market)

Databricks visualization. Run in Databricks to view.

In [0]:
zero_profile = (
    silver_df
    .groupBy(
        "country",
        "coffee_type"
    )
    .agg(
        F.sum("zero_flag")
        .alias("zero_years"),

        F.count("*")
        .alias("observations")
    )
    .withColumn(
        "zero_pct",
        F.round(
            (
                F.col("zero_years")
                /
                F.col("observations")
            ) * 100,
            2
        )
    )
    .filter(
        F.col("zero_years") > 0
    )
    .orderBy(
        F.desc("zero_years")
    )
)

display(zero_profile)

Databricks visualization. Run in Databricks to view.

In [0]:
all_zero_series = (
    silver_df
    .groupBy(
        "country",
        "coffee_type"
    )
    .agg(
        F.max(
            "domestic_consumption"
        ).alias("max_consumption")
    )
    .filter(
        F.col("max_consumption") == 0
    )
)

display(all_zero_series)